In [6]:
import pandas as pd
import numpy as np
import datetime as dt

In [19]:

def load_raw_data(filepath):
    raw_df = pd.read_excel(filepath, header = [0,1])
    return raw_df



In [20]:
def clean_data_date(raw_df):
    """
    Input : raw DataFrame
    Output: cleaned DataFrame —
            missing values handled, duplicates removed,
            sorted chronologically, consistent schema enforced
    """
    
    #-----------------Date-------------------#
    ##create new column to test check which date cannot be parsed successfully
    raw_df[('real_date','real_date')] = pd.to_datetime(raw_df[('Unnamed: 0_level_0','Dates')], errors = 'coerce')
    raw_df.drop(('Unnamed: 0_level_0','Dates'), axis = 1)
    
    ##-------check any wrong date format--------##
    print(f'sum of NaN in dates = {raw_df[('real_date','real_date')].isna().sum()}')

    ##----------sort datetime----------#
    raw_df = raw_df.sort_values(('real_date', 'real_date'))
    
    ##------drop rows with duplicate date and ticker--------##
    raw_df = raw_df.drop_duplicates(subset = [('real_date', 'real_date')])
    
    ##-------set date as index---------#
    raw_df = raw_df.set_index(('real_date', 'real_date'))
    
    #-------Drop original date---------#
    raw_df = raw_df.drop(("Unnamed: 0_level_0","Dates"), axis = 1)
    
    return raw_df
    
    
    
  

In [21]:
res = clean_data_date(load_raw_data('toy_market_data.xlsx')).head()
print(res)


sum of NaN in dates = 1
                       TICK001 US Equity                            \
                                 PX_LAST     PX_VOLUME     PX_OPEN   
(real_date, real_date)                                               
2024-01-01                    100.248357  9.572576e+05   99.952150   
2024-01-02                    100.179225  9.880408e+05  100.156000   
2024-01-03                    100.503069  9.981741e+05  100.199297   
2024-01-04                    100.000000  1.000000e+06   99.800000   
2024-01-05                    101.147507  1.031275e+06  101.114982   

                                               TICK002 US Equity  \
                            PX_LOW     PX_HIGH           PX_LAST   
(real_date, real_date)                                             
2024-01-01               99.041500  101.523490        250.636867   
2024-01-02               98.950218  101.618724        250.490890   
2024-01-03               99.231799  101.897897        249.163302   
2024-01

In [22]:
def change_to_long(raw_df):
    return raw_df.stack(level = 0)


res_2 = (change_to_long(res))
print(res_2)



                                             PX_LAST     PX_VOLUME  \
(real_date, real_date)                                               
2024-01-01             TICK001 US Equity  100.248357  9.572576e+05   
                       TICK002 US Equity  250.636867  4.936775e+05   
                       TICK003 US Equity   49.776743  2.039083e+06   
                       TICK004 US Equity  300.476006  7.852857e+05   
2024-01-02             TICK001 US Equity  100.179225  9.880408e+05   
                       TICK002 US Equity  250.490890  4.963192e+05   
                       TICK003 US Equity   50.204942  2.003511e+06   
                       TICK004 US Equity  301.202281  8.348274e+05   
2024-01-03             TICK001 US Equity  100.503069  9.981741e+05   
                       TICK002 US Equity  249.163302  4.886940e+05   
                       TICK003 US Equity   50.311989  2.140880e+06   
                       TICK004 US Equity  301.383288  8.230225e+05   
2024-01-04          

In [25]:
def check_type(long_raw_df):
    cols = ['PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'PX_VOLUME']
    original_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    long_raw_df[cols] = long_raw_df[cols].apply(pd.to_numeric, errors = 'coerce')
    
    after_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    ##-----NaN difference between original and after--------#
    print(original_null - after_null)
    long_raw_df.index = long_raw_df.index.set_names(['real_date', 'ticker'])
    
    return long_raw_df
    

check_type(res_2)
# res_3 = check_type(res_2)
# print(res_3)


PX_OPEN      0
PX_HIGH      0
PX_LOW       0
PX_LAST      0
PX_VOLUME    0
dtype: int64


PX_LAST     PX_VOLUME     PX_OPEN  \
real_date  ticker                                                    
2024-01-01 TICK001 US Equity  100.248357  9.572576e+05   99.952150   
           TICK002 US Equity  250.636867  4.936775e+05  250.392140   
           TICK003 US Equity   49.776743  2.039083e+06   49.331740   
           TICK004 US Equity  300.476006  7.852857e+05  300.000975   
2024-01-02 TICK001 US Equity  100.179225  9.880408e+05  100.156000   
           TICK002 US Equity  250.490890  4.963192e+05  249.998065   
           TICK003 US Equity   50.204942  2.003511e+06   50.035944   
           TICK004 US Equity  301.202281  8.348274e+05  300.726977   
2024-01-03 TICK001 US Equity  100.503069  9.981741e+05  100.199297   
           TICK002 US Equity  249.163302  4.886940e+05  249.042275   
           TICK003 US Equity   50.311989  2.140880e+06   50.124197   
           TICK004 US Equity  301.383288  8.230225e+05  301.096569   
2024-01-04 TICK001 US Equity  100.000000  1.000000e+06   99.800000   
           TICK002 US Equity         NaN           NaN         NaN   
           TICK003 US Equity   50.000000  2.000000e+06   49.800000   
           TICK004 US Equity  300.000000  8.000000e+05  299.800000   
2024-01-05 TICK001 US Equity  101.147507  1.031275e+06  101.114982   
           TICK002 US Equity  249.138303  4.877628e+05  248.757493   
           TICK003 US Equity   49.775710  2.054584e+06   49.486570   
           TICK004 US Equity  301.359094  7.815837e+05  301.134872   

                                  PX_LOW     PX_HIGH  
real_date  ticker                                     
2024-01-01 TICK001 US Equity   99.041500  101.523490  
           TICK002 US Equity  249.356991  251.827805  
           TICK003 US Equity   48.722388   50.914172  
           TICK004 US Equity  299.507345  301.060112  
2024-01-02 TICK001 US Equity   98.950218  101.618724  
           TICK002 US Equity  249.348859  251.377626  
           TICK003 US Equity   49.093221   51.431033  
           TICK004 US Equity  300.645977  302.403250  
2024-01-03 TICK001 US Equity   99.231799  101.897897  
           TICK002 US Equity  248.579162  250.600032  
           TICK003 US Equity   49.392389   51.787841  
           TICK004 US Equity  300.764470  301.956051  
2024-01-04 TICK001 US Equity   99.000000  101.000000  
           TICK002 US Equity         NaN         NaN  
           TICK003 US Equity   49.000000   51.000000  
           TICK004 US Equity  299.000000  301.000000  
2024-01-05 TICK001 US Equity  100.289042  102.569382  
           TICK002 US Equity  247.739749  249.979370  
           TICK003 US Equity   48.919737   50.598666  
           TICK004 US Equity  300.209884  302.565336

In [14]:

def drop_non_universal(long_raw_df):
      #------------PRICE--------------#
    #1.check if it is under universal 100 in that year
    check_universal = long_raw_df.copy()

    check_universal = check_universal.reset_index()
    
    check_universal['year'] = check_universal['real_date'].dt.year
    
    ##proportion of non-null/all < 0.5 -> drop
    check_universal = check_universal.groupby(['ticker', 'year']).apply(lambda x: x.count()/(x.count()+x.isnull().sum())).map(lambda x: x<0.5)
    
    check_universal['to_drop'] = check_universal.any(axis = 1)
    
    check_universal = check_universal.reset_index()
    
    to_drop_list = check_universal[check_universal['to_drop']]['ticker'].tolist()
    
    print(to_drop_list)
    
    long_raw_df = long_raw_df[~long_raw_df.index.get_level_values('ticker').isin(to_drop_list)]
    
    return long_raw_df
    



In [15]:
# res_3 = drop_non_universal(res_2)
# print(res_3)
test = pd.read_csv('toy_half_blank_ticker.csv')
test['real_date'] = pd.to_datetime(test['real_date'])
# test.dtypes
drop_non_universal(test)

['TICK_HALF_BLANK']


KeyError: 'Requested level (ticker) does not match index name (None)'

In [201]:
def check_price_and_volume(long_raw_df):
    
    
    # print(long_raw_df)
    long_raw_df.loc[lambda x: ~((x['PX_LOW'] < x['PX_LAST']) & (x['PX_LAST']  < x['PX_HIGH']) & (x['PX_LOW'] < x['PX_OPEN']) & (x['PX_OPEN']< x['PX_HIGH'])), ['PX_LOW', 'PX_HIGH', 'PX_OPEN', 'PX_LAST']] = None
    
 
    ##---------Track unusual volume with Z-score------------##
    
    mean = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('mean')
    std = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('std')
    long_raw_df['z_score'] = (long_raw_df['PX_VOLUME'] - mean) / std
    long_raw_df.loc[lambda x: x['z_score'].abs() > 3, 'PX_VOLUME'] = None
    long_raw_df = long_raw_df.drop('z_score', axis = 1)
    
    long_raw_df = long_raw_df.ffill(limit = 3)
    
    
    return long_raw_df

res_4 = check_price_and_volume(res_3)

print(res_4)

                                 PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0 

In [ ]:
def check_misalign_date(long_raw_data):
    long_raw_data = long_raw_data.reset_index()

    date_counts = long_raw_data.groupby('real_date')['ticker'].count()
    correct_date =  date_counts[date_counts == date_counts.max()]
    
    correct_date = correct_date.index.tolist()
    
    
    long_raw_data = long_raw_data[long_raw_data['real_date'].isin(correct_date)]
    
    long_raw_data = long_raw_data.set_index(['real_date', 'ticker'])
    
    return long_raw_data

check_misalign_date(res_5)

    

PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0   78.103672   78.034933   
           NDX Index          105.178921  2314785.0  105.037270  104.630062   
2020-01-07 TICK002 US Equity  298.194698  2058532.0  299.152835  297.357725   
           TICK004 US Equity   77.869288   862784.0   77.984328   77.782379   
           NDX Index          105.782631  1212384.0  105.804051  105.549470   

                                 PX_HIGH  
real_date  ticker                         
2020-01-01 TICK002 US Equity  283.429203  
           TICK004 US Equity   78.475302  
           NDX Index          102.621097  
2020-01-02 TICK002 US Equity  291.488938  
           TICK004 US Equity   77.686614  
           NDX Index          103.922327  
2020-01-03 TICK002 US Equity  291.797436  
           TICK004 US Equity   79.198394  
           NDX Index          105.457421  
2020-01-06 TICK002 US Equity  297.300806  
           TICK004 US Equity   78.474943  
           NDX Index          105.473471  
2020-01-07 TICK002 US Equity  299.499712  
           TICK004 US Equity   78.772573  
           NDX Index          106.265215

In [230]:
def add_return_columns(aligned_df):
    """
    Input : aligned price DataFrame
    Output: same DataFrame + simple return and log return columns
            (basic derived series only — no risk/strategy metrics here)
    """
    aligned_df['simple_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform('pct_change')
    
    aligned_df['log_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform(lambda x : np.log(x/x.shift(1)))
    
    aligned_df[['simple_return', 'log_return']] = aligned_df[['simple_return', 'log_return']].fillna(0)
    
    return aligned_df

add_return_columns(check_misalign_date(res_5))

PX_LAST  PX_VOLUME     PX_OPEN      PX_LOW  \
real_date  ticker                                                             
2020-01-01 TICK002 US Equity  283.053872   915460.0  283.378931  281.846567   
           TICK004 US Equity   78.154937   773189.0   78.313159   78.086426   
           NDX Index          101.483077   395246.0  102.093013  100.838939   
2020-01-02 TICK002 US Equity  289.696332  2532517.0  290.124047  288.883081   
           TICK004 US Equity   77.617135  1610706.0   77.355708   77.278543   
           NDX Index          103.374469  2543394.0  103.673107  103.318950   
2020-01-03 TICK002 US Equity  291.278370   542786.0  289.943629  288.821998   
           TICK004 US Equity   79.102443  2687682.0   79.193192   78.603322   
           NDX Index          105.288720   711083.0  105.385887  105.221662   
2020-01-06 TICK002 US Equity  295.412626  1010006.0  295.572586  293.714884   
           TICK004 US Equity   78.064690  2040634.0   78.103672   78.034933   
           NDX Index          105.178921  2314785.0  105.037270  104.630062   
2020-01-07 TICK002 US Equity  298.194698  2058532.0  299.152835  297.357725   
           TICK004 US Equity   77.869288   862784.0   77.984328   77.782379   
           NDX Index          105.782631  1212384.0  105.804051  105.549470   

                                 PX_HIGH  simple_return  log_return  
real_date  ticker                                                    
2020-01-01 TICK002 US Equity  283.429203       0.000000    0.000000  
           TICK004 US Equity   78.475302       0.000000    0.000000  
           NDX Index          102.621097       0.000000    0.000000  
2020-01-02 TICK002 US Equity  291.488938       0.023467    0.023196  
           TICK004 US Equity   77.686614      -0.006881   -0.006905  
           NDX Index          103.922327       0.018638    0.018466  
2020-01-03 TICK002 US Equity  291.797436       0.005461    0.005446  
           TICK004 US Equity   79.198394       0.019136    0.018956  
           NDX Index          105.457421       0.018518    0.018348  
2020-01-06 TICK002 US Equity  297.300806       0.014193    0.014094  
           TICK004 US Equity   78.474943      -0.013119   -0.013206  
           NDX Index          105.473471      -0.001043   -0.001043  
2020-01-07 TICK002 US Equity  299.499712       0.009418    0.009374  
           TICK004 US Equity   78.772573      -0.002503   -0.002506  
           NDX Index          106.265215       0.005740    0.005723

In [2]:
"""
datalayer.py
Loads, cleans, and validates historical market data.
No analysis, no strategy logic, no risk metrics — just trustworthy data out.
"""
import pandas as pd
import numpy as np


##-------1. load data --------##

def load_raw_data(filepath):
    raw_df = pd.read_excel(filepath, header = [0,1])
    return raw_df

##-------2. load clean_data_date --------##

def clean_data_date(raw_df):
    """
    Input : raw DataFrame
    Output: cleaned DataFrame —
            missing values handled, duplicates removed,
            sorted chronologically, consistent schema enforced
    """
    #-----------------Date-------------------#
    ##create new column to test check which date cannot be parsed successfully
    raw_df[('real_date','real_date')] = pd.to_datetime(raw_df[('Unnamed: 0_level_0','Dates')], errors = 'coerce')
    raw_df = raw_df.drop(('Unnamed: 0_level_0','Dates'), axis = 1)
    
    ##-------check any wrong date format--------##
    print(f'sum of NaN in dates = {raw_df[('real_date','real_date')].isna().sum()}')

    ##----------sort datetime----------#
    raw_df = raw_df.sort_values(('real_date', 'real_date'))
    
    ##------drop rows with duplicate date and ticker--------##
    raw_df = raw_df.drop_duplicates(subset = [('real_date', 'real_date')])
    
    ##-------set date as index---------#
    raw_df = raw_df.set_index(('real_date', 'real_date'))
    
    #-------Drop original date---------#
    # raw_df = raw_df.drop(("Unnamed: 0_level_0","Dates"), axis = 1)
    
    return raw_df


##-------3. change_to_long data --------##
    
def change_to_long(raw_df):
    return raw_df.stack(level = 0)


##-------4. check type mismatch --------##

def check_type(long_raw_df):
    cols = ['PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'PX_VOLUME']
    original_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    long_raw_df[cols] = long_raw_df[cols].apply(pd.to_numeric, errors = 'coerce')
    
    after_null = long_raw_df[cols].apply(lambda x: x.isnull().sum())
    
    ##-----NaN difference between original and after--------#
    print(original_null - after_null)
    
    long_raw_df.index = long_raw_df.index.set_names(['real_date', 'ticker'])
    return long_raw_df
    


##-------5. drop non universal ticker --------##

def drop_non_universal(long_raw_df):
      #------------PRICE--------------#
    #1.check if it is under universal 100 in that year
    
    
    check_universal = long_raw_df.copy()

    check_universal = check_universal.reset_index()
    
    ##proportion of non-null/all < 0.5 -> drop
    check_universal = check_universal.drop('real_date', axis = 1).groupby('ticker').apply(lambda x: x.count()/(x.count()+x.isnull().sum())).map(lambda x: x<0.5)
    
    check_universal['to_drop'] = check_universal.any(axis = 1)
    
    to_drop_list = check_universal[check_universal['to_drop']].index.tolist()
    
    long_raw_df = long_raw_df[~long_raw_df.index.get_level_values('ticker').isin(to_drop_list)]
    
    return long_raw_df


##-------6. check misalign date --------##

def check_misalign_date(long_raw_data):
    long_raw_data = long_raw_data.reset_index()

    date_counts = long_raw_data.groupby('real_date')['ticker'].count()
    correct_date =  date_counts[date_counts == date_counts.max()]
    
    correct_date = correct_date.index.tolist()
    
    long_raw_data = long_raw_data[long_raw_data['real_date'].isin(correct_date)]
    
    long_raw_data = long_raw_data.set_index(['real_date', 'ticker'])
    
    return long_raw_data


##-------7. track unusual price and volume --------##

def check_price_and_volume(long_raw_df):
    # print(long_raw_df)
    long_raw_df.loc[lambda x: ~((x['PX_LOW'] < x['PX_LAST']) & (x['PX_LAST']  < x['PX_HIGH']) & (x['PX_LOW'] < x['PX_OPEN']) & (x['PX_OPEN']< x['PX_HIGH'])), ['PX_LOW', 'PX_HIGH', 'PX_OPEN', 'PX_LAST']] = None
    ##---------Track unusual volume with Z-score------------##
    
    mean = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('mean')
    std = long_raw_df.groupby('ticker')['PX_VOLUME'].transform('std')
    long_raw_df['z_score'] = (long_raw_df['PX_VOLUME'] - mean) / std
    long_raw_df.loc[lambda x: x['z_score'].abs() > 3, 'PX_VOLUME'] = None
    long_raw_df = long_raw_df.drop('z_score', axis = 1)
    
    long_raw_df = long_raw_df.groupby('ticker').ffill(limit = 3)
    
    return long_raw_df

##-------8. add return columns --------##
def add_return_columns(aligned_df):
    """
    Input : aligned price DataFrame
    Output: same DataFrame + simple return and log return columns
            (basic derived series only — no risk/strategy metrics here)
    """
    aligned_df['simple_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform('pct_change')
    
    aligned_df['log_return'] = aligned_df.groupby('ticker')['PX_LAST'].transform(lambda x : np.log(x/x.shift(1)))
    
    aligned_df[['simple_return', 'log_return']] = aligned_df[['simple_return', 'log_return']].fillna(0)
    
    return aligned_df



def validate_data(final_df):
    """
    Input : fully processed DataFrame
    Output: pass/fail or list of issues —
            checks for leftover NaNs, mismatched row counts,
            out-of-range values, unexpected date gaps
    """
    issues = []
    
    null_counts = final_df.isnull().sum()
    if null_counts.sum() > 0:
        issues.append(f"NaNs remaining (expected due to ffill limit=3):\n{null_counts[null_counts > 0]}")
    
    row_counts = final_df.groupby('ticker').size()
    if row_counts.nunique() > 1:
        issues.append(f"Mismatched row counts:\n{row_counts}")
    
    if (final_df[['PX_LAST','PX_OPEN','PX_HIGH','PX_LOW']] < 0).any().any():
        issues.append("Negative prices found")
    
    if issues:
        for i in issues:
            print(i)
    else:
        print("All checks passed")
        
        
#---------RUN PIPELINE ---------#

def run_pipeline(filepath):
    print("\n--- 1. Load raw data ---")
    df = load_raw_data(filepath)
    print(df.shape)

    print("\n--- 2. Clean dates ---")
    df = clean_data_date(df)
    print(df.shape)

    print("\n--- 3. Convert to long format ---")
    df = change_to_long(df)
    print(df.shape)
    print(df.head())

    print("\n--- 4. Check/fix column types ---")
    df = check_type(df)
    

    print("\n--- 5. Drop non-universal tickers ---")
    before = df.index.get_level_values('ticker').nunique()
    df = drop_non_universal(df)
    after = df.index.get_level_values('ticker').nunique()
    print(f"Tickers: {before} -> {after}")

    print("\n--- 6. Fix misaligned dates ---")
    before = df.shape[0]
    df = check_misalign_date(df)
    after = df.shape[0]
    print(f"Rows: {before} -> {after}")

    print("\n--- 7. Check price/volume anomalies ---")
    df = check_price_and_volume(df)

    print("\n--- 8. Add return columns ---")
    df = add_return_columns(df)
    print(df[['PX_LAST', 'simple_return', 'log_return']].head())

    print("\n--- 9. Validate final data ---")
    issues = validate_data(df)

    print("\n--- Done ---")
    return df, issues

In [3]:
df = load_raw_data("NDX_Universe_OHLC.xlsx")
print(df.shape)

(1691, 506)


In [4]:
df = clean_data_date(df)
print(df.shape)

sum of NaN in dates = 0
(1691, 505)


/var/folders/ny/bdyt6s3j1cdcqxf5yttkd2x00000gn/T/ipykernel_84622/769466698.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_df[('real_date','real_date')] = pd.to_datetime(raw_df[('Unnamed: 0_level_0','Dates')], errors = 'coerce')


In [5]:
print("\n--- 3. Convert to long format ---")
df = change_to_long(df)
print(df.shape)
print(df.head())


--- 3. Convert to long format ---
(170791, 5)
                                       PX_LAST    PX_VOLUME  PX_OPEN  \
(real_date, real_date)                                                 
2020-01-01             NDX Index       8733.07  112989658.0  8681.33   
                       ABNB US Equity      NaN          NaN      NaN   
                       ADBE US Equity   329.81    1592512.0   326.25   
                       ADI US Equity    118.84     963456.0   118.82   
                       ADP US Equity    170.50     903032.0   169.22   

                                         PX_LOW    PX_HIGH  
(real_date, real_date)                                      
2020-01-01             NDX Index       8674.380  8735.4300  
                       ABNB US Equity       NaN        NaN  
                       ADBE US Equity   326.250   329.9400  
                       ADI US Equity    118.135   119.1407  
                       ADP US Equity    169.220   170.6600  


In [6]:
print("\n--- 4. Check/fix column types ---")
df = check_type(df)


--- 4. Check/fix column types ---
PX_OPEN      0
PX_HIGH      0
PX_LOW       0
PX_LAST      0
PX_VOLUME    0
dtype: int64


In [7]:
print("\n--- 5. Drop non-universal tickers ---")
before = df.index.get_level_values('ticker').nunique()
df = drop_non_universal(df)
after = df.index.get_level_values('ticker').nunique()
print(f"Tickers: {before} -> {after}")


--- 5. Drop non-universal tickers ---
Tickers: 101 -> 96


In [8]:
print("\n--- 6. Fix misaligned dates ---")
before = df.shape[0]
df = check_misalign_date(df)
after = df.shape[0]
print(f"Rows: {before} -> {after}")


--- 6. Fix misaligned dates ---
Rows: 162336 -> 162336


In [9]:
print("\n--- 7. Check price/volume anomalies ---")
df = check_price_and_volume(df)


--- 7. Check price/volume anomalies ---


In [12]:
print("\n--- 8. Add return columns ---")
df = add_return_columns(df)
print(df[['PX_LAST', 'simple_return', 'log_return']][:105])


--- 8. Add return columns ---
                           PX_LAST  simple_return  log_return
real_date  ticker                                            
2020-01-01 NDX Index       8733.07       0.000000    0.000000
           ABNB US Equity      NaN       0.000000    0.000000
           ADBE US Equity      NaN       0.000000    0.000000
           ADI US Equity    118.84       0.000000    0.000000
           ADP US Equity       NaN       0.000000    0.000000
...                            ...            ...         ...
2020-01-02 ADP US Equity    170.32       0.000000    0.000000
           ADSK US Equity   187.83       0.023820    0.023541
           AEP US Equity     93.46      -0.011110   -0.011172
           ALNY US Equity   115.62       0.003907    0.003900
           AMAT US Equity    62.20       0.019004    0.018826

[105 rows x 3 columns]


In [13]:
print("\n--- 9. Validate final data ---")
issues = validate_data(df)


--- 9. Validate final data ---
NaNs remaining (expected due to ffill limit=3):
PX_LAST      3177
PX_VOLUME    3341
PX_OPEN      3177
PX_LOW       3177
PX_HIGH      3177
dtype: int64
